# Distance to the training data

Fig 4 (end probe) and Fig 9 (mid probe): per question, Δe = |conf − accuracy| − |conf − consistency| against the embedding distance between the question and the training dataset. Final models, every (training, test) dataset pair, test split. Panel A colours points by training set, panel B by test set.

In [ ]:
import glob
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
from scipy import stats
from notebook_helper import *

emb_dist = load_embedding_distances(split="test")   # emb_dist[train][test]: distance per question


def question_points(probe):
    """One row per (training dataset, test question): distance and Δe, averaged over models x runs."""
    err_acc, err_cons = load_question_errors(emb_dist, probes=[probe], split="test")
    df = pd.concat([pd.DataFrame({"train_label": SHORT_LABELS[a], "test_label": SHORT_LABELS[e],
                                  "emb_dist": emb_dist[a][e],
                                  "delta": err_acc[a][e] - err_cons[a][e]})
                    for a in DATASETS for e in err_acc[a]], ignore_index=True).dropna()
    return df.sample(frac=1, random_state=0).reset_index(drop=True)   # random draw order

In [ ]:
# ICLR text width (5.5 in), 8-pt text; style after Cecchi & Palminteri (PNAS 2026)
FAMILY = {"math4": "MATH", "math10": "MATH", "mmlupro4": "MMLU-Pro", "mmlupro10": "MMLU-Pro",
          "medmcqa4": "MedMCQA"}
# Paper palette; MATH in red as in Fig 3 (similar test topics), the others in blues
FAMILY_COLORS = {"MATH": COLORS["red"], "MMLU-Pro": COLORS["steel"], "MedMCQA": COLORS["navy"]}
INK, INK_SOFT, PILL = "#222222", "#555555", "#E8E8E8"
XLIM, YLIM = (0.1, 0.9), (-0.4, 0.4)
FIGSIZE = (5.5, 2.1)                                              # inches
AXES = {"A": [0.115, 0.19, 0.411, 0.619], "B": [0.571, 0.19, 0.411, 0.619]}   # figure fraction

# DM Sans (SIL OFL) from ./fonts if available; DejaVu Sans otherwise, and for Greek glyphs
for f in glob.glob(os.path.join(REPO_ROOT, "fonts", "*.ttf")):
    font_manager.fontManager.addfont(f)
has_dm_sans = "DM Sans" in {f.name for f in font_manager.fontManager.ttflist}
DIST_RC = {
    "font.family": (["DM Sans"] if has_dm_sans else []) + ["DejaVu Sans"],
    "font.size": 7.5, "axes.labelsize": 8,
    "axes.edgecolor": INK_SOFT, "axes.linewidth": 0.8, "axes.labelcolor": INK,
    "xtick.color": INK_SOFT, "ytick.color": INK_SOFT, "xtick.labelcolor": INK, "ytick.labelcolor": INK,
    "xtick.direction": "out", "ytick.direction": "out",
    "xtick.major.size": 3.5, "ytick.major.size": 3.5, "xtick.minor.size": 2, "ytick.minor.size": 2,
    "xtick.major.width": 0.8, "ytick.major.width": 0.8,
    "pdf.fonttype": 42, "svg.fonttype": "path",     # SVG text as outlines: no font needed
}


def panel(ax, df, fit, colour_by, title, y_labels):
    ax.scatter(df["emb_dist"], df["delta"], c=df[colour_by].map(FAMILY_COLORS).to_numpy(),
               s=1.2, alpha=0.12, linewidths=0, rasterized=True, zorder=1)
    ax.axhline(0, color=INK_SOFT, lw=0.8, ls=(0, (4, 3)), zorder=2)
    xs = np.array([df["emb_dist"].min(), df["emb_dist"].max()])
    ax.plot(xs, fit.intercept + fit.slope * xs, color=INK, lw=1.2, zorder=3)
    ax.text(xs[1] + 0.01, fit.intercept + fit.slope * xs[1] + 0.035, f"r = {fit.rvalue:.2f}",
            ha="right", va="bottom", fontsize=7, color=INK, zorder=4)

    ax.set_xlim(*XLIM)
    ax.set_ylim(*YLIM)
    ax.set_xticks([0.2, 0.4, 0.6, 0.8])
    ax.set_xticks(np.arange(0.1, 0.901, 0.1), minor=True)
    ax.set_yticks([-0.4, 0, 0.4], labels=["−0.4", "0", "0.4"])
    ax.set_yticks(np.arange(-0.4, 0.401, 0.1), minor=True)
    ax.spines[["top", "right"]].set_visible(False)

    h = ax.get_position().height * FIGSIZE[1]                   # axes height, inches
    if y_labels:                                                # shared y axis: panel A only
        ax.text(0.0, 1 + 0.05 / h, "Δe", transform=ax.transAxes, ha="center", va="bottom",
                fontsize=8, color=INK)
        for y, label in [(0.75, "Closer to\nconsistency"), (0.25, "Closer to\naccuracy")]:
            ax.text(-0.19, y, label, transform=ax.transAxes, rotation=90, ha="center",
                    va="center", fontsize=8, color=INK, linespacing=1.1)
    else:
        ax.tick_params(axis="y", labelleft=False)
    ax.text(0.5, 1 + 0.22 / h, title, transform=ax.transAxes, ha="center", va="center",
            fontsize=8, fontweight="semibold", color=INK,
            bbox=dict(boxstyle="round,pad=0.45,rounding_size=0.9", fc=PILL, ec="none"))
    handles = [plt.Line2D([], [], ls="", marker="o", ms=4, mfc=c, mec="none", label=f)
               for f, c in FAMILY_COLORS.items()]
    ax.legend(handles=handles, loc="upper left", frameon=False, fontsize=7, handletextpad=0.15,
              borderaxespad=0.15, labelspacing=0.25)


def plot_distance(df, name):
    """Panel A coloured by training set, panel B by test set; fig/<name>.pdf/.svg/.png."""
    df = df.assign(train_family=df["train_label"].map(FAMILY),
                   test_family=df["test_label"].map(FAMILY))
    fit = stats.linregress(df["emb_dist"], df["delta"])
    with plt.rc_context(DIST_RC):
        fig = plt.figure(figsize=FIGSIZE)
        for letter, colour_by, title in [("A", "train_family", "Coloured by training set"),
                                         ("B", "test_family", "Coloured by test set")]:
            ax = fig.add_axes(AXES[letter])
            panel(ax, df, fit, colour_by, title, y_labels=(letter == "A"))
            h = AXES[letter][3] * FIGSIZE[1]
            x = -0.6 / (AXES[letter][2] * FIGSIZE[0]) if letter == "A" else -0.07
            ax.text(x, 1 + 0.22 / h, letter, transform=ax.transAxes, fontsize=12, color=INK,
                    va="center", ha="left")
        # one x label for both panels
        fig.text((AXES["A"][0] + AXES["B"][0] + AXES["B"][2]) / 2, 0.02,
                 f"Distance to training data (cosine, mean of the {N_NEAREST} nearest training items)",
                 ha="center", va="bottom", fontsize=8, color=INK)
        os.makedirs(FIG_PATH, exist_ok=True)
        for ext, dpi in (("pdf", 600), ("svg", 600), ("png", 300)):   # dpi of the rasterised points
            fig.savefig(os.path.join(FIG_PATH, f"{name}.{ext}"), dpi=dpi)
    plt.show()

    n_out = int(((df["delta"] < YLIM[0]) | (df["delta"] > YLIM[1])).sum())
    print(f"n = {len(df):,} points | r = {fit.rvalue:.3f} | slope = {fit.slope:.3f} ± {fit.stderr:.3f} "
          f"| intercept = {fit.intercept:.3f} | outside y-range: {n_out} ({100 * n_out / len(df):.2f}%)")

## Fig 4 — End probe

In [ ]:
plot_distance(question_points("end"), "fig04_distance_end")

## Fig 9 — Mid probe

In [ ]:
plot_distance(question_points("mid"), "fig09_distance_mid")